# Project 5 — Customer Churn Prediction & Analytics

**Goal:** Build an end-to-end machine-learning project that predicts whether a telecom customer will churn.

**Pipeline:** Raw CSV → Data cleaning → EDA → Feature engineering → Train/test split → Logistic Regression + Random Forest → Evaluation → Feature importance → Error analysis → Saved model → SQLite/SQL → Streamlit dashboard.

In [ ]:
print("Project 5: Customer Churn Prediction")
print("Starting the project...")

In [ ]:
# If needed, run this cell once:
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib streamlit

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

BASE = Path.cwd()
DATA_DIR = BASE / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = BASE / "models"

for p in [RAW_DIR, PROCESSED_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE)

## 1. Download and load the dataset

This project uses the IBM Telco Customer Churn dataset. The notebook downloads the CSV directly into `data/raw/` if it is not already there.

In [ ]:
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cpd4.5/data/customer_churn/WA_FnUseC_TelcoCustomerChurn.csv"
raw_path = RAW_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not raw_path.exists():
    try:
        urllib.request.urlretrieve(DATA_URL, raw_path)
        print("Dataset downloaded.")
    except Exception as e:
        print("Automatic download failed.")
        print("Download the IBM Telco Customer Churn CSV and save it as:", raw_path)
        raise e
else:
    print("Dataset already exists:", raw_path)

df = pd.read_csv(raw_path)
print("Shape:", df.shape)
df.head()

## 2. Understand the data

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nTarget distribution:")
print(df["Churn"].value_counts())
print("\nTarget percentage:")
print(df["Churn"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].describe())

## 3. Clean the data

In [ ]:
# CustomerID is an identifier, not a useful predictive feature.
df = df.drop(columns=["customerID"]).copy()

# Convert blank TotalCharges values to NaN, then remove rows with missing target-critical data.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

df = df.dropna(subset=["TotalCharges"]).copy()
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Clean shape:", df.shape)
print(df["ChurnFlag"].value_counts())

## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
axes[0].set_title("Tenure vs Churn")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly Charges vs Churn")
plt.tight_layout()
plt.show()

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))

contract_churn[["Yes"]].plot(kind="bar", legend=False, figsize=(7,4))
plt.ylabel("Churn rate (%)")
plt.title("Churn Rate by Contract Type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
display(internet_churn.round(2))

## 5. Feature engineering

In [ ]:
# Useful customer-level features.
df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, np.nan)
df["AvgMonthlySpend"] = df["AvgMonthlySpend"].fillna(df["MonthlyCharges"])

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

df["HasPartnerOrDependents"] = (
    (df["Partner"] == "Yes") | (df["Dependents"] == "Yes")
).astype(int)

print(df[["AvgMonthlySpend", "TenureGroup", "HasPartnerOrDependents"]].head())

## 6. Prepare features and target

In [ ]:
target = "ChurnFlag"

drop_cols = ["Churn", target]
X = df.drop(columns=drop_cols)
y = df[target]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created.")

## 8. Logistic Regression

In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

logistic_pipeline.fit(X_train, y_train)
logistic_pred = logistic_pipeline.predict(X_test)
logistic_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, logistic_pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, logistic_prob), 4))

## 9. Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))

## 10. Compare models

In [ ]:
def model_metrics(name, y_true, pred, prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }

results = pd.DataFrame([
    model_metrics("Logistic Regression", y_test, logistic_pred, logistic_prob),
    model_metrics("Random Forest", y_test, rf_pred, rf_prob)
])

display(results.round(4))

In [ ]:
results.set_index("Model")[["Accuracy","Precision","Recall","F1","ROC-AUC"]].plot(
    kind="bar", figsize=(10,5), ylim=(0,1), title="Model Performance Comparison"
)
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Confusion matrix and ROC curve

In [ ]:
# Use Random Forest for the final saved model in this project.
final_model = rf_pipeline
final_pred = rf_pred
final_prob = rf_prob

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test, final_pred, ax=ax)
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, final_prob)
plt.title("Random Forest ROC Curve")
plt.tight_layout()
plt.show()

## 12. Feature importance

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
rf = final_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(9,6))
sns.barplot(data=importance_df.head(15), x="importance", y="feature")
plt.title("Top 15 Random Forest Features")
plt.tight_layout()
plt.show()

## 13. Error analysis

In [ ]:
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = final_pred
error_df["predicted_probability"] = final_prob
errors = error_df[error_df.actual != error_df.predicted].copy()

print("Incorrect predictions:", len(errors))
print("Error rate:", round(len(errors) / len(error_df), 4))
display(errors.head(10))

## 14. Save the final model

In [ ]:
model_path = MODEL_DIR / "customer_churn_model.joblib"
joblib.dump(final_model, model_path)
print("Saved model:", model_path)
print("Size (bytes):", model_path.stat().st_size)

## 15. Test the saved model on new customers

In [ ]:
loaded_model = joblib.load(model_path)

example_customer = X_test.iloc[[0]].copy()
example_prediction = loaded_model.predict(example_customer)[0]
example_probability = loaded_model.predict_proba(example_customer)[0,1]

print("Example prediction:", "Churn" if example_prediction == 1 else "No Churn")
print("Churn probability:", round(example_probability, 3))

## 16. Save processed data

In [ ]:
processed_path = PROCESSED_DIR / "customer_churn_clean.csv"
df.to_csv(processed_path, index=False)
print("Saved:", processed_path)

## 17. Store business analytics in SQLite

In [ ]:
DB_PATH = BASE / "customer_churn.db"

analytics_df = df.copy()
analytics_df["ChurnRateFlag"] = analytics_df["ChurnFlag"]

with sqlite3.connect(DB_PATH) as conn:
    analytics_df.to_sql("customers", conn, if_exists="replace", index=False)

queries = {
    "overall": """
        SELECT COUNT(*) AS customers,
               ROUND(AVG(ChurnFlag) * 100, 2) AS churn_rate_percent
        FROM customers
    """,
    "contract": """
        SELECT Contract,
               COUNT(*) AS customers,
               ROUND(AVG(ChurnFlag) * 100, 2) AS churn_rate_percent
        FROM customers
        GROUP BY Contract
        ORDER BY churn_rate_percent DESC
    """,
    "internet": """
        SELECT InternetService,
               COUNT(*) AS customers,
               ROUND(AVG(ChurnFlag) * 100, 2) AS churn_rate_percent
        FROM customers
        GROUP BY InternetService
        ORDER BY churn_rate_percent DESC
    """,
    "payment": """
        SELECT PaymentMethod,
               COUNT(*) AS customers,
               ROUND(AVG(ChurnFlag) * 100, 2) AS churn_rate_percent
        FROM customers
        GROUP BY PaymentMethod
        ORDER BY churn_rate_percent DESC
    """
}

for name, query in queries.items():
    print("\n", name.upper())
    display(pd.read_sql_query(query, conn))

conn.close()
print("Database:", DB_PATH)

## 18. Final project summary

In [ ]:
print("PROJECT 5 COMPLETE")
print("=" * 55)
print("Customers:", len(df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print(results.round(4).to_string(index=False))
print("Final model: Random Forest + preprocessing pipeline")
print("Saved model:", model_path)
print("SQLite database:", DB_PATH)
print("=" * 55)